# SupportIQ — Stage 2.1: Schema Enforcement & Data Quarantine

> **Goal:** Validate all rows against Pydantic schema. Malformed rows go to `data/interim/quarantine.jsonl` with an explicit reason—never silently dropped.


### 1. Setup & Environment

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from supportiq.data.load import load_raw_dataframe
from supportiq.data.validate import validate_dataframe, validate_single_record

print("Validation engine and schemas loaded.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Validation engine and schemas loaded.


### 2. Validate Full Dataset
Execute schema validation across all 26,872 raw records.

In [2]:
df = load_raw_dataframe()
quarantine_file = project_root / "data/interim/quarantine.jsonl"

valid_df, quarantined_list, summary = validate_dataframe(df, quarantine_path=quarantine_file)

print(f"Total processed:     {summary.total_processed:,}")
print(
    f"Valid records:       {summary.valid_count:,} ({summary.valid_count / summary.total_processed * 100:.2f}%)"
)
print(f"Quarantined records: {summary.quarantined_count:,} ({summary.quarantine_rate:.2f}%)")

{"timestamp": "2026-09-21T19:50:04.472397+00:00", "level": "INFO", "name": "supportiq.data.validate", "message": "Starting schema validation for 26872 records", "module": "validate", "line": 60}


{"timestamp": "2026-09-21T19:50:04.612188+00:00", "level": "INFO", "name": "supportiq.data.validate", "message": "Validation complete: 26872 valid, 0 quarantined (0.00% quarantine rate)", "module": "validate", "line": 92}


Total processed:     26,872
Valid records:       26,872 (100.00%)
Quarantined records: 0 (0.00%)


### 3. Verify Quarantine Safety on Malformed Records
Demonstrate that corrupted rows (empty instructions or missing fields) are caught and rejected.

In [3]:
test_bad_rows = [
    {
        "flags": "B",
        "instruction": "   ",
        "category": "ORDER",
        "intent": "cancel_order",
        "response": "Valid response.",
    },
    {
        "flags": "B",
        "instruction": "Valid instruction",
        "category": "ORDER",
        "intent": "cancel_order",
        "response": "No",
    },
]

for row in test_bad_rows:
    is_valid, reason, _ = validate_single_record(row)
    print(f"Row:      {row}")
    print(f"Valid:    {is_valid}")
    print(f"Rejected: {reason}")
    print("-" * 60)

Row:      {'flags': 'B', 'instruction': '   ', 'category': 'ORDER', 'intent': 'cancel_order', 'response': 'Valid response.'}
Valid:    False
Rejected: instruction: Value error, Field cannot be empty or pure whitespace.
------------------------------------------------------------
Row:      {'flags': 'B', 'instruction': 'Valid instruction', 'category': 'ORDER', 'intent': 'cancel_order', 'response': 'No'}
Valid:    False
Rejected: response: String should have at least 5 characters
------------------------------------------------------------
